In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'mps' if torch.backends.mps.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
#eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

mps


-------  
**Why is it called bigram language model?**  
When we have a character we predict the second one, thus focusing on two characters at a time. Each prediction only depends on the one character right before it  that pairing (character, next character) is the "bigram."  

In [3]:
with open('sherlock_holmes.txt', 'r', encoding= 'utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)
print(vocab_size)

['\n', ' ', '!', '&', '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '£', '½', 'à', 'â', 'æ', 'è', 'é', 'œ', '—', '‘', '’', '“', '”']
88


----  
**Tokenizer -> Encoder + Decoder**  
Encoder converts each element of the array into an integer  
*we're using a character level tokenizer here.*

In [4]:
#generator loops
string_to_int = {ch : i for i, ch in enumerate(chars)} #string to integer mapping
int_to_string = {i : ch for i, ch in enumerate(chars)} #integer to string mapping
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([41, 56, 53,  1, 22, 52, 70, 53, 62, 68, 69, 66, 53, 67,  1, 63, 54,  1,
        40, 56, 53, 66, 60, 63, 51, 59,  1, 29, 63, 60, 61, 53, 67,  0,  0, 50,
        73,  1, 22, 66, 68, 56, 69, 66,  1, 24, 63, 62, 49, 62,  1, 25, 63, 73,
        60, 53,  0,  0,  0, 24, 63, 62, 68, 53, 62, 68, 67,  0,  0,  1,  1,  1,
        30,  8,  1,  1,  1,  1,  1, 22,  1, 40, 51, 49, 62, 52, 49, 60,  1, 57,
        62,  1, 23, 63, 56, 53, 61, 57, 49,  0])


In [5]:
#training and validation split
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y
x, y = get_batch('train')
print('inputs: ')
# print x.shape
print(x)
print('targets: ')
print(y)

inputs: 
tensor([[50, 73,  1, 56, 57, 67,  1, 60],
        [61, 53,  2,  1, 57, 68, 85, 67],
        [64, 63, 67, 67, 53, 67, 67, 57],
        [ 1, 30,  1, 56, 49, 52,  1, 66]], device='mps:0')
targets: 
tensor([[73,  1, 56, 57, 67,  1, 60, 63],
        [53,  2,  1, 57, 68, 85, 67,  1],
        [63, 67, 67, 53, 67, 67, 57, 63],
        [30,  1, 56, 49, 52,  1, 66, 53]], device='mps:0')


In [6]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

#sequential -> CPU
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print('when input is', context, 'target is', target)

when input is tensor([41]) target is tensor(56)
when input is tensor([41, 56]) target is tensor(53)
when input is tensor([41, 56, 53]) target is tensor(1)
when input is tensor([41, 56, 53,  1]) target is tensor(22)
when input is tensor([41, 56, 53,  1, 22]) target is tensor(52)
when input is tensor([41, 56, 53,  1, 22, 52]) target is tensor(70)
when input is tensor([41, 56, 53,  1, 22, 52, 70]) target is tensor(53)
when input is tensor([41, 56, 53,  1, 22, 52, 70, 53]) target is tensor(62)


In [7]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

---  
Block size - size of sequence  
Batch size - how many sequences/blocks we can process parallely  
*without these you can't expect fast training and good performance.*

In [8]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # gonna be a learnable para because of the nn
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

        # forward pass function helps us understand the underlying ops of the model.
    def forward(self, index, targets = None):
        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self.forward(index)
            # focus only on the last time step
            logits = logits[:, -1, :] # become (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim = -1) # (B, C)
            # sample from the distribution
            index_next = torch.multinomial(probs, num_samples = 1) # (B, 1)
            # append sampled index to the running sequence
            index = torch.cat((index, index_next), dim = 1) # (B, T + 1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype = torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens = 500)[0].tolist())
print(generated_chars)


B6qè½ANSjkS
Bu12(ui4œF,.dO’.;KIèC&cmgsâYgqdLcQa&’Q½“x4)zYoB.BNW;bxmJm‘UUqYi
éaB
h2q½0T&3£B.é3tvc 3éhstxMAOS—5O(l1QW 3cf&tàotnW‘wlZKz5Qr-H”?&V0 3½6DmNoqJJuv4CrèD:97FC127tEFeæ”ZgA—l_01æ”’j-YO!èD?tPâ£39pHKâzRPbL‘£èSDtâWw4YO“kT’iom:Z1é3TM—5eOQHtàNh !v&2AcfHU.n“Ivnm‘i!Wvx5k0tE:6d—om0‘lIa2DémScd—5èNàp.t’VIapLp.RzémE;Xf½m:6_b‘UPHf6ffDéR)pHz?q&K.i6.y5nWl——nDFj)½Mo8rèDcVTo;mTPæ7sx“”—(GHæC ?A?RmR-W3pPXxdw,)bpl“2rFnæ18HTD5j4è‘j-’y”Q4—tW3.dRMt£5kèj.iYG‘éhtxdâJlOd,Bk!£nWzN7q?zYjeU9r,?GS1&kC)fpruVl3w0Lmkà‘PBu


In [9]:
# create a PyTorch optimiser
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

for iter in range(max_iters):
    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses['train']:.4f}, val loss: {losses['val']:.4f}')
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.8901, val loss: 4.8876
step: 250, train loss: 4.8320, val loss: 4.8238
step: 500, train loss: 4.7632, val loss: 4.7546
step: 750, train loss: 4.6959, val loss: 4.6868
4.655092716217041


In [10]:
context = torch.zeros((1, 1), dtype = torch.long, device = device)
generated_chars = decode(m.generate(context, max_new_tokens = 500)[0].tolist())
print(generated_chars)


B’”è78£”éwcmomGMAV6_Q-3”cQ4—oAEsRNx“-K)Wucf6 sNE18.”½0S
âjH’½œu.X (?54FZ£Khd—yoD(He
Z½Byv:M9ot:!v(Mq”BLâTmœ‘iM:6SxkKkOsR8Izèj)l”kVpM:1vuE&HE1MVWà4td—u8’dnS-.i(eFN0yv.qy7i0næ(QS5DV_lvc2½0yVQlfzE7c59Lj“,t;GYR_”&V.D?”I800tS5nM3V?q(Ex“ BYO(Ex“—&vpbC4(3dg (pYC”T9,?9bC3œuY1N;(u.XzN7wA2vu5Yi6TPLckcyvzYi9zU£7rnWqotFuLœè&”-qJ“&àvcvcQWYtypo_2BklD gèrèLAss1è“-.è7,o7W?XVp&JkræqHnfmxàUh4âx“d8zgâgœu!m_8sXCHUBV127½Qvd2jgeJoVO5‘Vy(ovS,9IWC½9ci(;XX’m,)—nuv6RGCèOB.q—j½?o?RbEx5apQvsWæg &pu00(zYgsW3&URM
yv.vGKg;£;!
